# LightGBM — Prédiction des consommations énergétiques

## Split train/test

Split partagé avec `clustering_stratifie.ipynb` (`idx_train.npy` / `idx_test.npy` / `stratum_final`),
identique à celui utilisé dans `lightgbm_stratified.ipynb` et `mlp_stratified.ipynb`. Les jeux de
features globales (`X`) et physiques (`X_physical`) sont découpés avec exactement le même
`idx_test` : la comparaison Global vs Physique porte donc sur les mêmes bâtiments de test.

*(Avant cette révision, ce notebook construisait sa propre stratification par `qcut` et splittait
séparément les features globales et physiques, ce qui rendait la comparaison Global vs Physique
non comparable — chaque jeu de features était évalué sur un jeu de test différent.)*

## Variables cibles

Le modèle prédit simultanément les cinq variables suivantes :

- `out.electricity.total.energy_consumption..kwh`
- `out.emissions.total.lrmer_mid_case_25..co2e_kg`
- `out.natural_gas.total.energy_consumption..kwh`
- `out.fuel_oil.total.energy_consumption..kwh`
- `out.propane.total.energy_consumption..kwh`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb

ROOT = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / "data" / "processed"

X = pd.read_parquet(DATA_PROCESSED / "X.parquet")
Y = pd.read_parquet(DATA_PROCESSED / "Y.parquet")
X_physical = pd.read_parquet(DATA_PROCESSED / "X_physical_engineered.parquet")

print(X.shape)
print(X_physical.shape)

In [ ]:
# ============================================================
# Split partagé avec clustering_stratifie.ipynb (idx_train / idx_test / stratum_final)
# ============================================================

idx_train = np.load(DATA_PROCESSED / "idx_train.npy")
idx_test  = np.load(DATA_PROCESSED / "idx_test.npy")

labels = pd.read_parquet(DATA_PROCESSED / "cluster_labels.parquet")
stratum_train = labels.iloc[idx_train]["stratum_final"]

# X_physical est construit à partir de X.copy() (même index, même ordre de lignes) :
# le même idx_train / idx_test s'applique donc directement aux deux jeux de features.
assert X.index.equals(X_physical.index), "X et X_physical n'ont pas le même index"

print("idx_train / idx_test :", idx_train.shape, idx_test.shape)

In [ ]:
def build_splits(X_data, idx_train, idx_test, stratum_train, test_size=0.2, random_state=42):
    """Découpe train / val / test à partir du split partagé (idx_train/idx_test)
    et de la stratification déjà calculée (stratum_train) pour le sous-split val."""
    X_train = X_data.iloc[idx_train]
    X_test = X_data.iloc[idx_test]
    Y_train = Y.iloc[idx_train]
    Y_test = Y.iloc[idx_test]

    X_train_final, X_val, Y_train_final, Y_val = train_test_split(
        X_train, Y_train,
        test_size=test_size,
        random_state=random_state,
        stratify=stratum_train,
    )
    return X_train_final, X_val, X_test, Y_train_final, Y_val, Y_test


X_train, X_val, X_test, Y_train, Y_val, Y_test = build_splits(
    X, idx_train, idx_test, stratum_train
)
X_train_physical, X_val_physical, X_test_physical, Y_train_physical, Y_val_physical, Y_test_physical = build_splits(
    X_physical, idx_train, idx_test, stratum_train
)

total = len(X_train) + len(X_val) + len(X_test)
print(f"Train : {X_train.shape} ({len(X_train)/total:.1%})")
print(f"Val   : {X_val.shape} ({len(X_val)/total:.1%})")
print(f"Test  : {X_test.shape} ({len(X_test)/total:.1%})")

In [ ]:
results_dummy = {}

for target in Y.columns:

    median = Y_train[target].median()
    y_pred = np.full(len(Y_test), median)

    results_dummy[target] = {
        "RMSE": np.sqrt(mean_squared_error(Y_test[target], y_pred)),
        "MAE": mean_absolute_error(Y_test[target], y_pred),
        "R2": r2_score(Y_test[target], y_pred)
    }

pd.DataFrame(results_dummy).T


In [ ]:
models = {}
predictions = {}

for target in Y.columns:

    print(f"\n========== {target} ==========")

    models[target] = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=5,
        random_state=42
    )

    models[target].fit(
        X_train,
        Y_train[target],
        eval_set=[(X_val, Y_val[target])],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    y_pred = models[target].predict(X_test)
    predictions[target] = y_pred

In [ ]:
models_physical = {}
predictions_physical = {}

for target in Y.columns:

    print(f"\n========== {target} ==========")

    models_physical[target] = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=5,
        random_state=42
    )

    models_physical[target].fit(
        X_train_physical,
        Y_train_physical[target],
        eval_set=[(X_val_physical, Y_val_physical[target])],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    y_pred_physical = models_physical[target].predict(X_test_physical)
    predictions_physical[target] = y_pred_physical

In [ ]:
results_pct = {}

for target in Y.columns:
    y_true = Y_test[target]
    y_pred = predictions[target]

    mean = y_true.mean()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    results_pct[target] = {
        "RMSE": rmse,
        "Erreur RMSE_%": rmse / mean * 100,
        "MAE": mae,
        "Erreur MAE_%": mae / mean * 100,
        "R2": r2_score(y_true, y_pred),
    }

results_df = (
    pd.DataFrame(results_pct)
    .T
    .sort_values("RMSE", ascending=True)
)
results_df

In [ ]:
results_pct_physical = {}

for target in Y.columns:
    y_true = Y_test_physical[target]
    y_pred_physical = predictions_physical[target]

    mean = y_true.mean()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_physical))
    mae = mean_absolute_error(y_true, y_pred_physical)

    results_pct_physical[target] = {
        "RMSE": rmse,
        "Erreur RMSE_%": rmse / mean * 100,
        "MAE": mae,
        "Erreur MAE_%": mae / mean * 100,
        "R2": r2_score(y_true, y_pred_physical),
    }

results_df_physical = (
    pd.DataFrame(results_pct_physical)
    .T
    .sort_values("RMSE", ascending=True)
)
results_df_physical